# W13 · Why predict noise?

Advanced Machine Learning for Artificial Intelligence · November 24, 2026

Select Runtime → Change runtime type → CPU if needed. Run from top to bottom. The notebook generates its own 2D data and needs no upload, credentials or GPU. These activities are formative.

Inspect the forward marginal, train a small noise predictor and compare its generated distribution with held-out data. Labels are used only to interpret the plots.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.8.0",
                           "--index-url", "https://download.pytorch.org/whl/cpu"])
if importlib.util.find_spec("matplotlib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "matplotlib==3.10.3"])

get_ipython().run_line_magic("matplotlib", "inline")


## Setup · reusable CPU functions

The next cell embeds the course implementation so the notebook also runs when uploaded directly to Colab. The schedule arrays use time indices 0 through T. Add-noise accepts point and noise tensors of shape [batch, 2] and a vector of integer times. The predictor receives coordinates and three time features. Its two outputs are predicted Gaussian noise coordinates.


In [ ]:
"""Small CPU DDPM for a two-component Gaussian mixture."""

from dataclasses import dataclass
from math import pi

import torch
from torch import nn
from torch.nn.functional import mse_loss


@dataclass(frozen=True)
class Schedule:
    """One-based DDPM coefficients with an identity at index zero."""

    beta: torch.Tensor
    alpha: torch.Tensor
    alpha_bar: torch.Tensor
    posterior_variance: torch.Tensor

    @property
    def steps(self):
        return len(self.beta) - 1


def linear_schedule(steps=160):
    """Return CPU coefficients with beta increasing from 0.0001 to 0.08."""
    beta = torch.cat((torch.zeros(1), torch.linspace(0.0001, 0.08, steps)))
    alpha = 1.0 - beta
    alpha_bar = alpha.cumprod(0)
    variance = torch.zeros_like(beta)
    variance[1:] = beta[1:] * (1.0 - alpha_bar[:-1]) / (1.0 - alpha_bar[1:])
    return Schedule(beta, alpha, alpha_bar, variance)


def gaussian_mixture(n, seed):
    """Sample n points [n,2] and component IDs [n], using a private CPU RNG."""
    rng = torch.Generator().manual_seed(seed)
    centers = torch.tensor([[-1.8, 0.0], [1.8, 0.0]])
    component = torch.randint(0, len(centers), (n,), generator=rng)
    points = centers[component] + 0.32 * torch.randn(n, 2, generator=rng)
    return points, component, centers


def add_noise(clean, time, epsilon, schedule):
    """Draw the forward marginal using clean/epsilon [batch,2] and time [batch]."""
    signal_power = schedule.alpha_bar[time, None]
    return signal_power.sqrt() * clean + (1.0 - signal_power).sqrt() * epsilon


class NoisePredictor(nn.Module):
    """Map each 2D noisy point and time to a predicted 2D noise vector."""

    def __init__(self, steps, width=64):
        super().__init__()
        self.steps = steps
        self.network = nn.Sequential(
            nn.Linear(5, width), nn.SiLU(),
            nn.Linear(width, width), nn.SiLU(),
            nn.Linear(width, 2),
        )

    def forward(self, points, time):
        normalized = time[:, None].to(points.dtype) / self.steps
        phase = (2.0 * pi) * normalized
        features = torch.cat((points, normalized, phase.sin(), phase.cos()), dim=1)
        return self.network(features)


def fit_predictor(train, schedule, updates=1800, batch_size=256, seed=24):
    """Train on random rows, time steps and noise; return the model and MSE trace."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = NoisePredictor(schedule.steps)
    rng = torch.Generator().manual_seed(seed + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    losses = []
    model.train()
    for _ in range(updates):
        indices = torch.randint(0, len(train), (batch_size,), generator=rng)
        time = torch.randint(1, schedule.steps + 1, (batch_size,), generator=rng)
        clean = train[indices]
        epsilon = torch.randn(clean.shape, generator=rng)
        predicted = model(add_noise(clean, time, epsilon, schedule), time)
        loss = mse_loss(predicted, epsilon)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model.eval(), losses


@torch.no_grad()
def reverse_step(model, points, step, noise, schedule):
    """Return x_(t-1) [batch,2] from x_t and a fresh standard Gaussian draw."""
    time = torch.full((len(points),), step, dtype=torch.long)
    prediction = model(points, time)
    mean = (points - schedule.beta[step] * prediction /
            (1.0 - schedule.alpha_bar[step]).sqrt()) / schedule.alpha[step].sqrt()
    return mean + schedule.posterior_variance[step].sqrt() * noise


@torch.no_grad()
def generate(model, initial, schedule, seed):
    """Run all reverse steps from initial [batch,2] and return fresh 2D samples."""
    rng = torch.Generator().manual_seed(seed)
    points = initial
    for step in range(schedule.steps, 0, -1):
        noise = torch.randn(points.shape, generator=rng)
        points = reverse_step(model, points, step, noise, schedule)
    return points


def mode_counts(points, centers, radius=0.70):
    """Count points within each of two disjoint center disks and outside both."""
    hits = torch.cdist(points, centers) < radius
    return [*hits.sum(dim=0).tolist(), (~hits.any(dim=1)).sum().item()]


In [ ]:
import json
import time as clock
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

torch.set_num_threads(1)
schedule = linear_schedule()
train, _, centers = gaussian_mixture(4096, seed=131)
holdout, holdout_labels, _ = gaussian_mixture(1200, seed=132)


def comparison_panels(point_sets, titles, colors, show_disks=False):
    """Plot 2D tensors on shared axes; optional disks use the coverage radius."""
    fig, axes = plt.subplots(1, len(point_sets), figsize=(12, 3.6),
                             sharex=True, sharey=True, layout="constrained")
    for ax, points, title, color in zip(axes, point_sets, titles, colors):
        xy = points.detach().numpy()
        ax.scatter(xy[:, 0], xy[:, 1], s=6, alpha=0.48, c=color, linewidths=0)
        if show_disks:
            for center in centers:
                ax.add_patch(Circle(tuple(center.tolist()), 0.70, fill=False,
                                    edgecolor="#b65327", lw=1.5, linestyle="--"))
        ax.set(title=title, xlabel="$x_1$", xlim=(-3.4, 3.4), ylim=(-3.0, 3.0), aspect="equal")
        ax.grid(alpha=0.15)
        ax.set_axisbelow(True)
    axes[0].set_ylabel("$x_2$")
    plt.show()


## A · Predict the effect of time

The forward marginal is $x_t=\sqrt{\bar{\alpha}_t}\,x_0+\sqrt{1-\bar{\alpha}_t}\,\epsilon$.
Predict how the space between the two clusters changes when you set probe_time to 40 or 120. Change the value and rerun the cell. The same Gaussian draw is shared across panels to isolate the effect of the coefficients.


In [ ]:
probe_time = 80
fixed_epsilon = torch.randn(holdout.shape, generator=torch.Generator().manual_seed(133))


def at(t):
    return add_noise(holdout, torch.full((len(holdout),), t, dtype=torch.long),
                     fixed_epsilon, schedule)


comparison_panels([holdout, at(probe_time), at(schedule.steps)],
                  ["Held-out data", f"Forward t = {probe_time}", f"Forward t = {schedule.steps}"],
                  ["#14776b", "#22658c", "#657987"])


## B · Fit a noise predictor

Each update samples clean points, one time per point, and Gaussian noise. The regression inputs are the corrupted points and times. The sampled noise supplies the targets. Adam updates the network weights using the mean of squared coordinate errors. Compare early and late windows of training loss and check a separately seeded held-out noise batch.


In [ ]:
start = clock.perf_counter()
model, losses = fit_predictor(train, schedule)
fit_seconds = clock.perf_counter() - start
early = sum(losses[:200]) / 200
late = sum(losses[-200:]) / 200

eval_rng = torch.Generator().manual_seed(134)
eval_time = torch.randint(1, schedule.steps + 1, (len(holdout),), generator=eval_rng)
eval_noise = torch.randn(holdout.shape, generator=eval_rng)
with torch.no_grad():
    heldout_mse = mse_loss(model(add_noise(holdout, eval_time, eval_noise, schedule),
                                 eval_time), eval_noise).item()
print(f"First 200 MSE {early:.3f}    Last 200 MSE {late:.3f}")
print(f"Held-out MSE {heldout_mse:.3f}    CPU fit {fit_seconds:.1f} s")


## C · Generate from Gaussian noise

Start with 1,200 draws from a standard Gaussian. At each reverse step the predicted noise determines the mean correction. The sampler adds random noise scaled by the reverse standard deviation. At the final step the variance is zero.

The dashed circles are radius 0.70 around the known centers. The same rule and denominator apply to all three samples. A point outside both circles counts as outside, even if it is closer to one center.


In [ ]:
initial = torch.randn(holdout.shape, generator=torch.Generator().manual_seed(135))
generated = generate(model, initial, schedule, seed=136)
coverage = {label: mode_counts(sample, centers) for label, sample in
            [("heldout", holdout), ("initial", initial), ("generated", generated)]}
comparison_panels([holdout, initial, generated],
                  ["Held-out data", "Initial Gaussian", "Generated sample"],
                  ["#14776b", "#657987", "#22658c"], show_disks=True)
print("Sample             Left    Right   Outside")
for label, counts in coverage.items():
    print(f"{label:16s} {counts[0]:6d} {counts[1]:8d} {counts[2]:9d}")


## D · Same state, two draws

Predict what happens when you pass the same point and time to the reverse-step function with two different Gaussian draws. Change trial_seed and rerun the cell. Which part of the result is shared? Repeat at time 1 and explain what changes.


In [ ]:
trial_seed = 137
trial_step = 80
one_point = at(trial_step)[:1]
trial_rng = torch.Generator().manual_seed(trial_seed)
z_a = torch.randn(one_point.shape, generator=trial_rng)
z_b = torch.randn(one_point.shape, generator=trial_rng)
print(reverse_step(model, one_point, trial_step, z_a, schedule))
print(reverse_step(model, one_point, trial_step, z_b, schedule))


In [ ]:
assert all(parameter.device.type == "cpu" for parameter in model.parameters())
assert not torch.equal(train[:len(holdout)], holdout)
assert torch.allclose(at(0), holdout) and schedule.alpha_bar[-1] < 0.003
assert schedule.posterior_variance[1] == 0 and schedule.posterior_variance[80] > 0
assert generated.shape == holdout.shape and torch.isfinite(generated).all()
assert early > late and heldout_mse < 0.65
assert all(sum(counts) == len(holdout) for counts in coverage.values())
assert min(coverage["generated"][:2]) > 240
assert sum(coverage["generated"][:2]) > sum(coverage["initial"][:2])
print("W13_EVIDENCE " + json.dumps({
    "device": "cpu", "torch": torch.__version__, "n": len(holdout),
    "steps": schedule.steps, "terminal_signal_power": schedule.alpha_bar[-1].item(),
    "updates": len(losses), "fit_seconds": round(fit_seconds, 3),
    "early_mse": early, "late_mse": late, "heldout_mse": heldout_mse,
    "counts": coverage,
}, sort_keys=True))
print("W13_CPU_RUNTIME_PASS")


## Reading

[Murphy, Probabilistic Machine Learning: Advanced Topics](https://probml.github.io/pml-book/book2.html), §25.1–25.2.
[Ho, Jain & Abbeel, Denoising Diffusion Probabilistic Models](https://arxiv.org/html/2006.11239#S3), equations 4, 11 and 14, and algorithms 1 and 2.
